# Problem 3: Why are orders delivered late?

About 8% of delivered orders arrive after the date the customer was promised at checkout. Late orders also get much worse reviews (avg 2.6 vs 4.3), so this matters for the customer experience.

What I want to find out in this notebook:
- where in the process late orders lose time (approval, seller, or carrier)
- which factors still matter after controlling for the others
- how much the late rate would drop if sellers handed over parcels on time

Steps: load the delivered orders from the SQL view `v_delivered`, add seller-to-customer distance, look at the trends, then fit a logistic regression. I also check it against a gradient boosting model, trained on older months and tested on newer ones.

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import chart_style as cs
cs.apply(); warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
con = sqlite3.connect(ROOT / "data" / "olist.db")
OUT = ROOT / "data" / "processed"

## 1. Load data and add distance

In [2]:
d = pd.read_sql("SELECT * FROM v_delivered", con)
geo = pd.read_sql("SELECT zip_prefix, lat, lng FROM geo_zip", con)

d = (d.merge(geo.add_prefix("c_"), left_on="customer_zip_code_prefix", right_on="c_zip_prefix", how="left")
      .merge(geo.add_prefix("s_"), left_on="seller_zip_code_prefix", right_on="s_zip_prefix", how="left"))

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

d["distance_km"] = haversine_km(d.s_lat, d.s_lng, d.c_lat, d.c_lng)
# Keep the analysis window with complete months and sane timelines
d = d[(d.purchase_month.between("2017-01", "2018-08"))
      & (d.seller_handling_days >= 0) & (d.carrier_transit_days >= 0)].copy()
print(f"{len(d):,} delivered orders | distance available for {d.distance_km.notna().mean():.1%}")
print(f"Late rate: {d.is_late.mean():.2%} | late seller handoff rate: {d.seller_late_handoff.mean():.2%}")

94,844 delivered orders | distance available for 99.5%
Late rate: 8.21% | late seller handoff rate: 9.03%


## 2. Monthly trend

In [3]:
m = pd.read_csv(OUT / "monthly_kpis.csv")
fig, ax = plt.subplots(figsize=(11, 4.6))
x = np.arange(len(m))
ax.plot(x, m.late_pct, color=cs.BRICK, marker="o", ms=5, label="Delivered late (% of orders)")
ax.plot(x, m.seller_late_handoff_pct, color=cs.GREEN, marker="o", ms=5, label="Seller missed carrier cut-off (%)")
ax.set_xticks(x, [p[2:] for p in m.purchase_month], rotation=0, fontsize=8.5)
ax.set_ylabel("% of delivered orders")
for idx, txt in [(10, "Black Friday\npeak (Nov-17)"), (14, "Feb-Mar 2018\nspike: carrier-side"), (19, "Aug-18: promise\nwindow cut to 15 days")]:
    ax.annotate(txt, (idx, m.late_pct[idx]), xytext=(0, 18), textcoords="offset points",
                ha="center", fontsize=8.5, color=cs.GREY)
ax.legend(loc="upper left")
ax.set_ylim(0, 27)
ax.set_title("Late deliveries by purchase month (2017-2018)")
cs.save(fig, "p3_monthly_late_trend")
m[["purchase_month","late_pct","seller_late_handoff_pct","avg_delivery_days","avg_promised_days"]].iloc[[10,13,14,17,19]]

,purchase_month,late_pct,seller_late_handoff_pct,avg_delivery_days,avg_promised_days
10,2017-11,14.31,12.87,15.16,22.56
13,2018-02,15.99,12.23,16.95,24.53
14,2018-03,21.36,8.51,16.30,22.03
17,2018-06,1.36,6.73,9.24,27.77
19,2018-08,10.39,9.09,7.73,15.19


The spikes don't all have the same cause:
- **Nov 2017 (Black Friday):** seller late handoffs went up together with the late rate, so this looks like sellers being overloaded.
- **Feb-Mar 2018:** late rate reached 21% but seller handoffs were mostly normal (8.5% in March). So the delay happened after pickup, on the carrier side.
- **Aug 2018:** delivery was actually the fastest of all months (7.7 days), but the promised window was shortened from ~28 to ~15 days, so more orders counted as late.

Note: orders from late August 2018 that were still in transit when the data was exported are missing, so August probably looks a bit better than it really was.

## 3. Time spent in each stage

In [4]:
stage = (d.groupby("is_late")[["approval_hours","seller_handling_days","carrier_transit_days"]].mean()
           .assign(approval_days=lambda t: t.approval_hours/24).drop(columns="approval_hours")
           [["approval_days","seller_handling_days","carrier_transit_days"]])
stage.index = ["On-time orders", "Late orders"]
fig, ax = plt.subplots(figsize=(10, 3.4))
left = np.zeros(2)
for col, color, lab in [("approval_days", cs.MUSTARD, "Payment approval"),
                        ("seller_handling_days", cs.ORANGE, "Seller handling"),
                        ("carrier_transit_days", cs.PLUM, "Carrier transit")]:
    ax.barh(stage.index, stage[col], left=left, color=color, height=0.55, label=lab, edgecolor="white", linewidth=2)
    for i, v in enumerate(stage[col]):
        if v > 1.2: ax.text(left[i] + v/2, i, f"{v:.1f}d", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    left += stage[col].values
for i, v in enumerate(left): ax.text(v + 0.3, i, f"{v:.1f} days", va="center", fontsize=9.5, color=cs.DARK)
ax.set_xlabel("Average days from purchase"); ax.grid(axis="y", visible=False)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ax.set_title("Average days in each stage: on-time vs late orders")
cs.save(fig, "p3_stage_decomposition")
stage.round(2)

,approval_days,seller_handling_days,carrier_transit_days
On-time orders,0.39,2.60,7.91
Late orders,0.50,5.33,25.69


In [5]:
d["dist_band"] = pd.cut(d.distance_km, [-1, 100, 500, 1000, 2000, 5000],
                        labels=["<100 km", "100-500", "500-1,000", "1,000-2,000", "2,000+"])
g = d.pivot_table(index="dist_band", columns="seller_late_handoff", values="is_late", aggfunc="mean", observed=True) * 100
n = d.groupby("dist_band", observed=True).size()

fig, ax = plt.subplots(figsize=(10, 4.2))
xx = np.arange(len(g)); w = 0.38
b1 = ax.bar(xx - w/2 - 0.01, g[0], w, color=cs.GREEN, label="Seller handed over on time")
b2 = ax.bar(xx + w/2 + 0.01, g[1], w, color=cs.ORANGE, label="Seller missed the carrier cut-off")
for bars in (b1, b2):
    for b in bars: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f"{b.get_height():.0f}%", ha="center", fontsize=8.5, color=cs.GREY)
ax.set_xticks(xx, [f"{i}\n(n={n[i]:,})" for i in g.index], fontsize=9)
ax.set_ylabel("% delivered late"); ax.grid(axis="x", visible=False); ax.set_ylim(0, 38)
ax.legend(loc="upper left")
ax.set_title("Late delivery rate by distance and seller handoff")
cs.save(fig, "p3_late_by_distance_handoff")
g.round(1)

seller_late_handoff,0,1
dist_band,,
<100 km,4.20,27.70
100-500,5.80,22.70
"500-1,000",7.00,24.10
"1,000-2,000",9.50,24.80
"2,000+",12.00,31.40


## 4. Late delivery vs review score

In [6]:
r = d.dropna(subset=["review_score"])
dist = (r.groupby("is_late").review_score.value_counts(normalize=True).unstack().mul(100))
dist.index = ["On time", "Late"]
fig, ax = plt.subplots(figsize=(10, 3.2))
colors = ["#c44536", "#e07a3f", "#d9d9d9", "#9bbf8a", "#3b7d4f"]  # red -> grey -> green
left = np.zeros(2)
for s, c in zip([1,2,3,4,5], colors):
    ax.barh(dist.index, dist[s], left=left, color=c, height=0.55, edgecolor="white", linewidth=2, label=f"{s} star")
    for i, v in enumerate(dist[s]):
        if v > 5: ax.text(left[i]+v/2, i, f"{v:.0f}%", ha="center", va="center", fontsize=9, color="white" if s in (1,2,5) else cs.DARK)
    left += dist[s].values
ax.set_xlim(0, 100); ax.set_xlabel("% of reviews"); ax.grid(axis="y", visible=False)
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.25))
ax.set_title("Review scores: on-time vs late orders")
cs.save(fig, "p3_review_distribution")
dist.round(1)

review_score,1.00,2.00,3.00,4.00,5.00
On time,6.60,2.60,8.00,20.40,62.40
Late,46.20,7.90,11.40,12.30,22.20


## 5. Logistic regression

I used logistic regression because the odds ratios are easy to explain to non-technical people ("a late handoff multiplies the odds of a late delivery by X"). The peak months are added as dummy variables so the seller effect doesn't get mixed up with seasonality.

In [7]:
mdl = d.dropna(subset=["distance_km", "product_weight_g"]).copy()
mdl["dist_100km"]   = mdl.distance_km / 100
mdl["weight_kg"]    = mdl.product_weight_g / 1000
mdl["freight_ratio"]= mdl.freight_value / mdl.items_value.clip(lower=1)
mdl["multi_seller"] = (mdl.n_sellers > 1).astype(int)
mdl["multi_item"]   = (mdl.n_items > 1).astype(int)
mdl["peak_nov17"]   = (mdl.purchase_month == "2017-11").astype(int)
mdl["peak_febmar18"]= mdl.purchase_month.isin(["2018-02", "2018-03"]).astype(int)
mdl["is_late"]      = mdl.is_late.astype(int)

formula = ("is_late ~ seller_late_handoff + dist_100km + same_state + promised_days + "
           "weight_kg + freight_ratio + multi_item + multi_seller + peak_nov17 + peak_febmar18")
logit = smf.logit(formula, data=mdl).fit(disp=False)
ors = pd.DataFrame({"odds_ratio": np.exp(logit.params), "ci_low": np.exp(logit.conf_int()[0]),
                    "ci_high": np.exp(logit.conf_int()[1]), "p_value": logit.pvalues}).drop("Intercept")
ors = ors.sort_values("odds_ratio")
print(f"n = {int(logit.nobs):,} | pseudo-R² = {logit.prsquared:.3f}")
ors.round(3)

n = 94,357 | pseudo-R² = 0.139


,odds_ratio,ci_low,ci_high,p_value
multi_seller,0.26,0.16,0.42,0.00
same_state,0.41,0.39,0.45,0.00
multi_item,0.82,0.75,0.90,0.00
promised_days,0.91,0.91,0.91,0.00
freight_ratio,0.95,0.87,1.03,0.23
weight_kg,1.03,1.02,1.03,0.00
dist_100km,1.07,1.07,1.08,0.00
peak_nov17,2.52,2.33,2.71,0.00
peak_febmar18,4.03,3.81,4.26,0.00
seller_late_handoff,5.24,4.93,5.56,0.00


In [8]:
labels = {"seller_late_handoff": "Seller missed carrier cut-off", "peak_febmar18": "Ordered Feb-Mar 2018",
          "peak_nov17": "Ordered Nov 2017 (Black Friday)", "multi_seller": "Order split across sellers",
          "dist_100km": "Distance (+100 km)", "multi_item": "More than 1 item", "weight_kg": "Weight (+1 kg)",
          "freight_ratio": "Freight / price ratio (+1.0)", "same_state": "Seller & customer in same state",
          "promised_days": "Promised window (+1 day)"}
p = ors.rename(index=labels)
fig, ax = plt.subplots(figsize=(10, 4.8))
y = np.arange(len(p))
col = [cs.BRICK if v > 1 else cs.GREEN for v in p.odds_ratio]
ax.hlines(y, p.ci_low, p.ci_high, color=col, linewidth=2)
ax.scatter(p.odds_ratio, y, color=col, s=60, zorder=3, edgecolor="white", linewidth=2)
for yi, v in zip(y, p.odds_ratio): ax.text(p.ci_high.iloc[yi] * 1.06, yi, f"{v:.2f}x", va="center", fontsize=9, color=cs.DARK)
ax.axvline(1, color=cs.GREY, linewidth=1, linestyle="--")
ax.set_xscale("log"); ax.set_yticks(y, p.index)
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
ax.xaxis.set_major_locator(FixedLocator([0.2, 0.5, 1, 2, 5])); ax.xaxis.set_major_formatter(FixedFormatter(["0.2x", "0.5x", "1x", "2x", "5x"])); ax.xaxis.set_minor_locator(NullLocator()); ax.grid(axis="y", visible=False)
ax.set_xlabel("Odds ratio for 'delivered late' (log scale, 95% CI)")
ax.set_title("Odds ratios for late delivery (logistic regression)")
cs.save(fig, "p3_odds_ratios")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p3_odds_ratios.png')

Quick check: does a more complex model do much better? I trained gradient boosting on orders up to Mar 2018 and tested both models on Apr-Aug 2018. The AUCs are close, so I'm comfortable using the logistic regression results.

In [9]:
feats = ["seller_late_handoff","dist_100km","same_state","promised_days","weight_kg",
         "freight_ratio","multi_item","multi_seller"]
train, test = mdl[mdl.purchase_month <= "2018-03"], mdl[mdl.purchase_month >= "2018-04"]
gbm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, random_state=42).fit(train[feats], train.is_late)
lr_t = smf.logit("is_late ~ " + " + ".join(feats), data=train).fit(disp=False)
res = pd.DataFrame({
    "model": ["Logistic regression", "Gradient boosting"],
    "test_AUC (Apr-Aug 2018)": [roc_auc_score(test.is_late, lr_t.predict(test)),
                                roc_auc_score(test.is_late, gbm.predict_proba(test[feats])[:, 1])]})
res

,model,test_AUC (Apr-Aug 2018)
0,Logistic regression,0.79
1,Gradient boosting,0.77


## 6. How many late orders start with a late handoff?

The odds ratio tells us the risk per order. But I also want to know how many of the late orders actually had a late handoff.

In [10]:
late = d[d.is_late == 1]
attr = pd.Series({"late orders": len(late),
                  "of which seller handed over late": int(late.seller_late_handoff.sum()),
                  "share caused upstream by seller": round(late.seller_late_handoff.mean(), 3),
                  "share with an on-time handoff (carrier / promise)": round(1 - late.seller_late_handoff.mean(), 3)})
attr

late orders                                         7,790.00
of which seller handed over late                    2,124.00
share caused upstream by seller                         0.27
share with an on-time handoff (carrier / promise)       0.73
dtype: float64

Only about 27% of late orders had a late handoff. So even though the handoff is the strongest risk factor per order, most late orders were handed to the carrier on time and the delay came afterwards. Fixing only the sellers would reduce the late rate by about 20% at most (see the what-if below).

## 7. What-if: fewer late handoffs

For a share of the late-handoff orders I set `seller_late_handoff = 0`, re-score them with the model and count how many late orders disappear. Then I convert that into avoided 1-2 star reviews, using the difference in 1-2 star rates between late and on-time orders. The output is saved to `data/processed/whatif_handoff.csv` and used in the Excel what-if sheet.

In [11]:
base_late = logit.predict(mdl).sum()
bad_gap = (r[r.is_late==1].review_score.le(2).mean() - r[r.is_late==0].review_score.le(2).mean())
months = mdl.purchase_month.nunique()
rows = []
for cut in [0.10, 0.25, 0.50, 0.75, 1.00]:
    sim = mdl.copy()
    idx = sim[sim.seller_late_handoff == 1].sample(frac=cut, random_state=7).index
    sim.loc[idx, "seller_late_handoff"] = 0
    avoided = base_late - logit.predict(sim).sum()
    rows.append({"handoff_reduction": cut, "late_orders_avoided": avoided,
                 "late_rate_new_pct": (base_late - avoided) / len(mdl) * 100,
                 "bad_reviews_avoided": avoided * bad_gap,
                 "late_orders_avoided_per_month": avoided / months})
whatif = pd.DataFrame(rows)
whatif.to_csv(OUT / "whatif_handoff.csv", index=False)
pd.DataFrame({"metric": ["baseline_late_rate_pct", "bad_review_gap_pp", "orders_in_model", "months"],
              "value": [base_late/len(mdl)*100, bad_gap*100, len(mdl), months]}).to_csv(OUT / "whatif_baseline.csv", index=False)
print(f"Baseline modelled late rate: {base_late/len(mdl):.2%} | 1-2 star gap late vs on-time: {bad_gap:.1%}")
whatif.round(2)

Baseline modelled late rate: 8.20% | 1-2 star gap late vs on-time: 44.9%


,handoff_reduction,late_orders_avoided,late_rate_new_pct,bad_reviews_avoided,late_orders_avoided_per_month
0,0.10,151.38,8.04,68.02,7.57
1,0.25,380.01,7.80,170.76,19.00
2,0.50,764.21,7.39,343.40,38.21
3,0.75,"1,144.86",6.99,514.45,57.24
4,1.00,"1,534.00",6.58,689.32,76.70


## Summary and recommendations

| Finding | What I'd do | Owner | KPI to track |
|---|---|---|---|
| Missing the carrier cut-off multiplies the odds of being late by ~5, but only ~27% of late orders had a late handoff. Fixing all of them takes the late rate from 8.2% to ~6.6% | Alert sellers 24h before `shipping_limit_date`; lower search ranking for sellers above 15% late handoff | Seller Ops | Late handoff % (target < 5%) |
| ~73% of late orders were handed over on time; late orders spent 25.7 days in transit vs 7.9 for on-time ones | Track transit time by lane every week and move volume to a second carrier on the worst lanes | Logistics | p90 transit days by lane |
| Aug 2018: promise window cut from 28 to 15 days, late rate went up although delivery was faster | Set promise dates per lane (e.g. 85th percentile of transit time) instead of one flat cut | Product / Data | Late % |
| 54% of late orders get 1-2 stars vs 9% of on-time ones | Send a delay message + small voucher when an order passes its promise date | CX | 1-2 star share of late orders |